# Medical Insurance Cost Prediction — Project Walkthrough

A complete, module-by-module tour of the ML pipeline that predicts annual insurance charges from six policyholder attributes.

## Tech Stack

| Layer | Technology |
|-------|------------|
| Language | Python 3.13 |
| ML | scikit-learn, XGBoost |
| Data | pandas, NumPy |
| Visualisation | Matplotlib |
| API | FastAPI + Uvicorn |
| Frontend | Next.js 15 (TypeScript) |
| Serialisation | joblib (model), JSON (metadata) |

## End-to-End Flow

```
insurance.csv
     │
     ▼
  [clean]          src/preprocessing.py  — dedup, normalise, validate, impute
     │
     ▼
  [engineer]       src/feature_engineering.py — 8 new domain features
     │
     ▼
  [train]          src/train.py — 9-10 models, Pipeline + ColumnTransformer
     │
     ▼
  [evaluate]       src/evaluate.py — RMSE/MAE/R², 5-fold CV, composite rank
     │
     ▼
  [save]           models/best_model.pkl  +  feature_info.json
     │
     ▼
  [predict]        src/predict.py — LRU-cached load, single + batch inference
     │
     ▼
  [API]            main.py — FastAPI  POST /api/predict  GET /api/metrics …
     │
     ▼
  [dashboard]      frontend/ — Next.js pages: predict, metrics, EDA, dataset
```


## 1. Setup

Add the project root to `sys.path` so all `src.*` imports resolve correctly, then import common libraries.


In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('Setup complete ✓')


## 2. `src/utils.py` — Foundation

`utils.py` is the **base layer imported by every other module**. It owns:

- **Path constants** — `ROOT_DIR`, `DATA_DIR`, `MODELS_DIR`, `OUTPUTS_DIR`, `EDA_DIR`, `EVAL_DIR`, `METRICS_DIR`
- **Dataset constants** — `TARGET_COLUMN='charges'`, `RANDOM_STATE=42`, feature name lists
- **Domain thresholds** — `BMI_OBESE_THRESHOLD=30.0`, `SMOKER_SURCHARGE_FACTOR=3.0`
- **Utilities** — `get_logger()` (no duplicate handlers), `load_raw_data()`, `save_json()`, `ensure_output_dirs()`

Nothing in `utils.py` imports from other `src/` modules — it is dependency-free within the project.


In [ ]:
from src.utils import ROOT_DIR, DATA_DIR, MODELS_DIR, OUTPUTS_DIR, EDA_DIR, EVAL_DIR
from src.utils import TARGET_COLUMN, RANDOM_STATE, BMI_OBESE_THRESHOLD, SMOKER_SURCHARGE_FACTOR
from src.utils import NUMERIC_FEATURES, CATEGORICAL_FEATURES, ALL_FEATURES
print(f'Root: {ROOT_DIR}')
print(f'Target: {TARGET_COLUMN}, Seed: {RANDOM_STATE}')
print(f'Numeric features: {NUMERIC_FEATURES}')
print(f'Categorical features: {CATEGORICAL_FEATURES}')
print(f'BMI obese threshold: {BMI_OBESE_THRESHOLD}')
print(f'Smoker surcharge factor: {SMOKER_SURCHARGE_FACTOR}')


In [ ]:
from src.utils import load_raw_data
raw = load_raw_data()
print(f'Shape: {raw.shape}')
print(f'Columns: {list(raw.columns)}')
raw.head(3)


In [ ]:
from src.utils import get_logger, save_json, ensure_output_dirs
log = get_logger('walkthrough')
log.info('Logger works — timestamped, named, no duplicate handlers')
ensure_output_dirs()
print('Output directories created/verified')


## 3. `src/preprocessing.py` — Data Cleaning

Cleaning runs as a **4-step sequential pipeline**:

| Step | Function | What it does |
|------|----------|--------------|
| 1 | `remove_duplicates(df)` | Drop exact-duplicate rows, reset index |
| 2 | `standardise_categoricals(df)` | Lowercase + strip strings, warn on unexpected values |
| 3 | `validate_ranges(df)` | Set out-of-range numeric values to `NaN` |
| 4 | `handle_missing(df)` | Median-fill numeric, mode-fill categorical, drop rows with missing target |

`clean_dataframe()` calls all four in order. After cleaning, `build_preprocessor()` returns a `ColumnTransformer` with two sub-pipelines:
- **Numeric**: `SimpleImputer(median)` → optionally `StandardScaler`
- **Categorical**: `SimpleImputer(most_frequent)` → `OneHotEncoder(drop='first')`

`split_data()` does a **stratified 80/20 split** keyed on `smoker + charge_quartile` so both the smoker ratio (~20%) and the charge distribution are balanced across train and test.


In [ ]:
from src.preprocessing import remove_duplicates
deduped = remove_duplicates(raw)
print(f'Before: {len(raw)} rows  After: {len(deduped)} rows  Removed: {len(raw)-len(deduped)}')


In [ ]:
from src.preprocessing import standardise_categoricals
normed = standardise_categoricals(deduped)
print('sex values:', sorted(normed['sex'].unique()))
print('smoker values:', sorted(normed['smoker'].unique()))
print('region values:', sorted(normed['region'].unique()))


In [ ]:
from src.preprocessing import validate_ranges
import pandas as pd
test_df = normed.copy()
test_df.loc[0, 'age'] = 999   # impossible value
test_df.loc[1, 'bmi'] = -5    # impossible value
result = validate_ranges(test_df)
print('NaN count after validation:', result[['age','bmi']].isna().sum().to_dict())


In [ ]:
from src.preprocessing import clean_dataframe
cleaned = clean_dataframe(raw)
print(f'Shape: {cleaned.shape}')
print(f'Missing values: {cleaned.isna().sum().sum()}')
print(f'Dtypes:\n{cleaned.dtypes}')


In [ ]:
from src.preprocessing import build_preprocessor
# Linear models get scaling, tree models don't
prep_scaled  = build_preprocessor(scale_numeric=True,  numeric_features=['age','bmi','children'], categorical_features=['sex','smoker','region'])
prep_noscale = build_preprocessor(scale_numeric=False, numeric_features=['age','bmi','children'], categorical_features=['sex','smoker','region'])
print('Scaled preprocessor steps:')
for name, transformer, cols in prep_scaled.transformers:
    print(f'  {name}: {[s[0] for s in transformer.steps]} -> {cols}')
print('\nTree preprocessor (no scaler):')
for name, transformer, cols in prep_noscale.transformers:
    print(f'  {name}: {[s[0] for s in transformer.steps]} -> {cols}')


In [ ]:
from src.preprocessing import split_data
from src.feature_engineering import engineer_features
enriched = engineer_features(cleaned)
X_train, X_test, y_train, y_test = split_data(enriched)
print(f'Train: {len(X_train)} rows  Test: {len(X_test)} rows')
print(f'Train smoker %: {(X_train["smoker"]=="yes").mean()*100:.1f}%')
print(f'Test  smoker %: {(X_test["smoker"]=="yes").mean()*100:.1f}%')
print(f'y_train — min: ${y_train.min():,.0f}  max: ${y_train.max():,.0f}  mean: ${y_train.mean():,.0f}')


## 4. `src/feature_engineering.py` — 8 New Features

All 8 features use **only input columns** (no target leakage) and are safe to apply identically at training and inference time.

| Feature | Type | Domain motivation |
|---------|------|-------------------|
| `age_group` | categorical | Young / middle_age / senior risk bands |
| `bmi_category` | categorical | WHO weight classification |
| `is_obese` | binary int | BMI ≥ 30 flag — strong non-linear threshold |
| `smoker_obese` | binary int | **Interaction**: obese smoker pays ~4× more than others |
| `age_bmi` | float | age × bmi / 1000 — compound metabolic risk |
| `has_children` | binary int | Any dependant on the policy |
| `family_size` | categorical | individual / small_family / large_family |
| `age_smoker` | float | age × smoker flag — older smokers pay sharply more |

The `smoker_obese` interaction is the **single most predictive engineered feature** — it captures the non-additive synergy between obesity and smoking that drives the highest insurance charges in the dataset.


In [ ]:
from src.feature_engineering import engineer_features
print(f'Before: {list(cleaned.columns)}')
enriched = engineer_features(cleaned)
new_cols = [c for c in enriched.columns if c not in cleaned.columns]
print(f'\nNew columns ({len(new_cols)}): {new_cols}')
print(f'After: {enriched.shape[1]} columns total')


In [ ]:
print('=== smoker_obese: the highest-charge group ===')
print(enriched.groupby('smoker_obese')['charges'].agg(['count','mean','median']).round(0))
print('\nAverage ratio — obese smoker vs rest:')
obese_smoker_mean = enriched[enriched['smoker_obese']==1]['charges'].mean()
rest_mean         = enriched[enriched['smoker_obese']==0]['charges'].mean()
print(f'  Obese smoker mean: ${obese_smoker_mean:,.0f}')
print(f'  Everyone else:     ${rest_mean:,.0f}')
print(f'  Ratio: {obese_smoker_mean/rest_mean:.1f}x')


In [ ]:
print('age_group distribution:')
print(enriched['age_group'].value_counts())
print('\nbmi_category distribution:')
print(enriched['bmi_category'].value_counts())
print('\nfamily_size distribution:')
print(enriched['family_size'].value_counts())


In [ ]:
from src.feature_engineering import get_all_feature_groups
num_feats, cat_feats = get_all_feature_groups(X_train)
print(f'Numeric  ({len(num_feats)}): {num_feats}')
print(f'Categorical ({len(cat_feats)}): {cat_feats}')


## 5. `src/train.py` — Training Pipeline

`train.py` orchestrates the full training run in **5 steps**:

1. `prepare_data()` — load → clean → engineer → split
2. `_get_model_registry()` — returns 9 (or 10 with XGBoost) estimators
3. `train_all_models()` — for each model: build Pipeline, fit, evaluate on test set, 5-fold CV
4. `select_best_model()` — composite rank score (RMSE 45% + MAE 35% + R² 20%)
5. `save_best_model()` — serialise winning Pipeline to `models/best_model.pkl`

**Why Pipeline?** Wrapping preprocessor + estimator in `sklearn.Pipeline` guarantees that:
- No data leakage (scaler fit on train only, applied to test)
- The exact same transform chain runs at inference time
- `cross_val_score` refits the full pipeline per fold

**`SCALED_MODELS`** — Linear Regression, Ridge, Lasso, and SVR use `StandardScaler`; tree models skip it (unnecessary and slightly slower).


In [ ]:
import sys; sys.path.insert(0, '..')
models = {
    'Linear Regression':     'No regularisation — baseline',
    'Ridge Regression':      'L2 reg, alpha by RidgeCV (LOO-CV)',
    'Lasso Regression':      'L1 reg, alpha by LassoCV (5-fold, 100 candidates)',
    'Decision Tree':         'max_depth=6, min_samples_leaf=10',
    'Random Forest':         '300 trees, max_depth=10, max_features=0.7',
    'Gradient Boosting':     '300 estimators, lr=0.03, subsample=0.8',
    'Support Vector Regressor': 'RBF kernel, C=5000, epsilon=200',
    'Extra Trees':           '300 trees, same config as RF',
    'AdaBoost':              '200 estimators, lr=0.05',
    'XGBoost':               '300 rounds, max_depth=4, lr=0.03',
}
print(f'{"Model":<35}  Config')
print('-'*70)
for name, cfg in models.items():
    print(f'{name:<35}  {cfg}')


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV
from src.preprocessing import build_preprocessor, split_data, clean_dataframe
from src.feature_engineering import engineer_features, get_all_feature_groups
from src.utils import load_raw_data, RANDOM_STATE

raw      = load_raw_data()
cleaned  = clean_dataframe(raw)
enriched = engineer_features(cleaned)
X_train, X_test, y_train, y_test = split_data(enriched)
num_feats, cat_feats = get_all_feature_groups(X_train)

# Build a Lasso pipeline (same way train.py does it)
lasso_pipeline = Pipeline(steps=[
    ('preprocessor', build_preprocessor(scale_numeric=True, numeric_features=num_feats, categorical_features=cat_feats)),
    ('regressor',    LassoCV(alphas=100, cv=5, max_iter=50_000, random_state=RANDOM_STATE)),
])

lasso_pipeline.fit(X_train, y_train)
print(f'CV-selected alpha: {lasso_pipeline["regressor"].alpha_:.4f}')
print(f'Input shape: {X_train.shape}  ->  After OHE: {lasso_pipeline["preprocessor"].transform(X_train).shape}')
print(f'Non-zero coefficients: {(lasso_pipeline["regressor"].coef_ != 0).sum()} / {len(lasso_pipeline["regressor"].coef_)}')


In [ ]:
import json
from src.utils import MODELS_DIR
feature_info = json.loads((MODELS_DIR / 'feature_info.json').read_text())
model_meta   = json.loads((MODELS_DIR / 'model_metadata.json').read_text())
print('feature_info.json:')
for k, v in feature_info.items():
    print(f'  {k}: {v}')
print('\nmodel_metadata.json:')
for k, v in model_meta.items():
    print(f'  {k}: {v}')


## 6. `src/evaluate.py` — Metrics & Selection

Four key functions:

| Function | Input | Output |
|----------|-------|--------|
| `compute_metrics(y_true, y_pred)` | arrays | dict: MAE, MSE, RMSE, R², timing |
| `evaluate_model(pipeline, ...)` | pipeline + train/test splits | (metrics dict, fitted pipeline, predictions) |
| `run_cross_validation(pipeline, X, y)` | unfitted pipeline | mean/std R² and RMSE over 5 folds |
| `select_best_model(metrics_df, cv_data)` | comparison DataFrame | best model name |

**Composite rank formula** (lower = better):
```
composite = 0.45 × rmse_rank + 0.35 × mae_rank + 0.20 × r2_rank
```
If two models are within 0.5 rank points, **CV R² breaks the tie** — preferring the model that generalises better across folds, not just the test set.


In [ ]:
from src.evaluate import compute_metrics
import numpy as np

y_pred = lasso_pipeline.predict(X_test)
metrics = compute_metrics(y_test.values, y_pred)
print(f'RMSE : ${metrics["rmse"]:>10,.2f}')
print(f'MAE  : ${metrics["mae"]:>10,.2f}')
print(f'R²   : {metrics["r2"]:>11.4f}')


In [ ]:
from src.evaluate import run_cross_validation
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV
from src.preprocessing import build_preprocessor

cv_pipe = Pipeline([
    ('preprocessor', build_preprocessor(True, num_feats, cat_feats)),
    ('regressor',    LassoCV(alphas=100, cv=5, max_iter=50_000, random_state=RANDOM_STATE)),
])
cv = run_cross_validation(cv_pipe, X_train, y_train)
print(f'5-fold CV R²:  {cv["r2_mean"]:.4f} ± {cv["r2_std"]:.4f}')
print(f'Per-fold R²:   {[round(v,4) for v in cv["r2_folds"]]}')
print(f'CV RMSE:       ${cv["rmse_mean"]:,.0f} ± ${cv["rmse_std"]:,.0f}')


In [ ]:
from src.evaluate import select_best_model
import pandas as pd, json
from src.utils import METRICS_DIR

comparison = pd.read_csv(METRICS_DIR / 'comparison.csv')
cv_data    = json.loads((METRICS_DIR / 'cross_validation.json').read_text())

# Show what the composite score looks like
df = comparison.copy()
df['rmse_rank'] = df['rmse'].rank(ascending=True)
df['mae_rank']  = df['mae'].rank(ascending=True)
df['r2_rank']   = df['r2'].rank(ascending=False)
df['composite'] = 0.45*df['rmse_rank'] + 0.35*df['mae_rank'] + 0.20*df['r2_rank']

ranking = df.sort_values('composite')[['model','rmse','mae','r2','composite']].reset_index(drop=True)
ranking.index += 1
ranking['rmse'] = ranking['rmse'].map('${:,.0f}'.format)
ranking['mae']  = ranking['mae'].map('${:,.0f}'.format)
ranking['r2']   = ranking['r2'].map('{:.4f}'.format)
ranking['composite'] = ranking['composite'].map('{:.2f}'.format)
print(ranking.to_string())


In [ ]:
disp = pd.read_csv(METRICS_DIR / 'comparison.csv').sort_values('rmse').reset_index(drop=True)
disp.index += 1
disp['rmse'] = disp['rmse'].map('${:,.0f}'.format)
disp['mae']  = disp['mae'].map('${:,.0f}'.format)
disp['r2']   = disp['r2'].map('{:.4f}'.format)
disp['train_time_sec'] = disp['train_time_sec'].map('{:.3f}s'.format)
print(disp[['model','rmse','mae','r2','train_time_sec']].to_string())


## 7. `src/predict.py` — Inference

Four public functions build the inference layer:

| Function | What it does |
|----------|--------------|
| `load_model(path)` | Load `best_model.pkl` — **LRU-cached**, so repeated calls are memory lookups |
| `_prepare_single(raw_dict, all_features)` | dict → DataFrame → `engineer_features()` → align columns |
| `predict_charges(raw_input)` | Single prediction — calls `_prepare_single` then `pipeline.predict()` |
| `predict_batch(df)` | Vectorised batch prediction over a DataFrame |

**LRU cache strategy**: `_load_model_cached` and `_load_feature_info_cached` are decorated with `@lru_cache(maxsize=16)`. The model pickle (~several MB) is read from disk exactly once per process; all subsequent calls hit memory. `clear_model_cache()` invalidates both caches — useful in notebooks and testing.


In [ ]:
from src.predict import predict_charges, get_model_info

profiles = [
    ('Young healthy non-smoker', {'age':22,'sex':'male',  'bmi':22.0,'children':0,'smoker':'no', 'region':'northeast'}),
    ('Middle-age with 2 kids',   {'age':35,'sex':'female','bmi':28.5,'children':2,'smoker':'no', 'region':'northwest'}),
    ('Obese smoker',             {'age':45,'sex':'male',  'bmi':33.0,'children':1,'smoker':'yes','region':'southeast'}),
    ('Senior obese smoker',      {'age':60,'sex':'female','bmi':35.0,'children':0,'smoker':'yes','region':'southwest'}),
]
print(f'{"Profile":<30}  Predicted Charge')
print('-'*50)
for label, profile in profiles:
    charge = predict_charges(profile)
    print(f'{label:<30}  ${charge:>10,.2f}')


In [ ]:
from src.predict import predict_batch
import pandas as pd

batch = pd.DataFrame([p for _, p in profiles])
results = predict_batch(batch)
batch['predicted'] = results.map('${:,.2f}'.format)
print(batch[['age','sex','bmi','smoker','predicted']].to_string(index=False))


In [ ]:
info = get_model_info()
print(f'Model name:      {info["model_name"]}')
print(f'Target column:   {info["target_column"]}')
print(f'Numeric ({len(info["numeric_features"])}):  {info["numeric_features"]}')
print(f'Categorical ({len(info["categorical_features"])}): {info["categorical_features"]}')
print(f'Total features:  {info["n_features_total"]}')


In [ ]:
import time
from src.predict import load_model, clear_model_cache

clear_model_cache()

t0 = time.perf_counter(); load_model(); t1 = time.perf_counter()
first_load = t1 - t0

t0 = time.perf_counter(); load_model(); t1 = time.perf_counter()
cached_load = t1 - t0

print(f'First load  (disk read): {first_load:.3f}s')
print(f'Cached load (memory):    {cached_load:.4f}s')
print(f'Speedup: {first_load/cached_load:.0f}x faster')


## 8. Lasso Coefficient Analysis

Lasso's L1 penalty drives irrelevant feature coefficients exactly to zero, performing built-in feature selection. After OHE expands the feature space, we can inspect which columns Lasso kept and which it eliminated.


In [ ]:
import joblib, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.utils import MODELS_DIR
from src.preprocessing import get_feature_names_out

bundle       = joblib.load(MODELS_DIR / 'best_model.pkl')
pipeline     = bundle['model']
regressor    = pipeline.named_steps['regressor']
preprocessor = pipeline.named_steps['preprocessor']

feature_names = get_feature_names_out(preprocessor)
coefs = regressor.coef_

coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
coef_df['abs'] = coef_df['coefficient'].abs()
coef_df = coef_df.sort_values('abs', ascending=False)

print(f'Total features after OHE: {len(coefs)}')
print(f'Non-zero (selected by Lasso): {(coefs != 0).sum()}')
print(f'Zero (eliminated by Lasso):   {(coefs == 0).sum()}')
print()
print('Top 10 features by coefficient magnitude:')
print(coef_df.head(10)[['feature','coefficient']].to_string(index=False))


In [ ]:
top = coef_df.head(12)
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#ef4444' if v < 0 else '#6366f1' for v in top['coefficient']]
ax.barh(top['feature'][::-1], top['coefficient'][::-1], color=colors[::-1], alpha=0.85)
ax.axvline(0, color='white', lw=0.8)
ax.set_xlabel('Coefficient (USD)')
ax.set_title('Lasso — Top 12 Feature Coefficients')
plt.tight_layout()
plt.show()


## 9. `main.py` — FastAPI Backend

The FastAPI server exposes **JSON endpoints only** — all page rendering is handled by Next.js.

### Endpoint Table

| Method | Path | Description |
|--------|------|-------------|
| `POST` | `/api/predict` | Predict charges for one policyholder |
| `GET` | `/api/metrics` | Model comparison leaderboard (sorted by RMSE) |
| `GET` | `/api/cv` | 5-fold cross-validation R² and RMSE per model |
| `GET` | `/api/dataset/summary` | Shape, target stats, smoker/region breakdowns |
| `GET` | `/api/dataset/sample` | First 20 rows as JSON records |
| `GET` | `/api/feature-info` | Best model name + feature lists |
| `GET` | `/api/eda-figures` | List of available EDA figure URLs |
| `GET` | `/api/eval-figures/{stem}` | Per-model evaluation plot URLs |
| `GET` | `/outputs/...` | Static PNG files (EDA + evaluation charts) |

### Pydantic Validation

```python
class PredictRequest(BaseModel):
    age:      int   = Field(..., ge=0,  le=120)
    sex:      str   # validated: 'male' | 'female'
    bmi:      float = Field(..., ge=10, le=80)
    children: int   = Field(..., ge=0,  le=10)
    smoker:   str   # validated: 'yes' | 'no'
    region:   str   # validated: northeast | northwest | southeast | southwest
```

Field validators lowercase and strip all string fields before they reach `predict_charges()`.

### Prediction Flow

```
POST /api/predict
  -> PredictRequest (Pydantic validates + normalises)
  -> predict_charges(inp)  [src/predict.py]
      -> load_model()        [LRU cache]
      -> _prepare_single()   [engineer_features + align columns]
      -> pipeline.predict()  [preprocessor + regressor]
  <- JSON: { predicted_charges, formatted, model_used, context, risk_flags }
```

### In-Memory Cache

The `_cache` dict holds `raw` (DataFrame), `metrics` (DataFrame), `cv` (dict), and `fi` (dict). These are loaded once on first request and reused for the lifetime of the process — the `/api/predict` endpoint is the only one that matters for latency, and it delegates to the `@lru_cache` in `src/predict.py`.

### CORS

```python
CORSMiddleware(
    allow_origins=["http://localhost:3000", "http://127.0.0.1:3000"],
    allow_methods=["*"],
    allow_headers=["*"],
)
```

Allows the Next.js dev server to call the API without browser CORS errors. Tighten `allow_origins` in production.


## 10. `frontend/` — Next.js Dashboard

### Page → API Endpoint Mapping

| Page (route) | File | FastAPI endpoints consumed |
|---|---|---|
| `/` (home) | `src/app/page.tsx` | none (static landing) |
| `/predict` | `src/app/predict/page.tsx` | `POST /api/predict` |
| `/metrics` | `src/app/metrics/page.tsx` | `GET /api/metrics`, `GET /api/cv` |
| `/eda` | `src/app/eda/page.tsx` | `GET /api/eda-figures`, `/outputs/figures/eda/*.png` |
| `/dataset` | `src/app/dataset/page.tsx` | `GET /api/dataset/summary`, `GET /api/dataset/sample` |
| `/about` | `src/app/about/page.tsx` | `GET /api/feature-info` |

### Key Components

**`Sidebar.tsx`**
- Persistent navigation rail present on every page
- Uses Next.js `Link` for client-side routing (no full page reload)
- Active state driven by `usePathname()` hook

**`ChartCard.tsx`**
- Reusable card wrapper for displaying chart images with title + description
- Accepts `title`, `description`, `imageUrl`, and `onClick` props
- Clicking opens the chart full-screen via `ImageModal`

**`ImageModal.tsx`**
- Lightbox overlay that renders a full-resolution PNG
- Controlled by `isOpen` / `onClose` props
- Renders into a portal to avoid z-index stacking issues
- Keyboard-accessible: pressing Escape closes the modal

### UI Component Library

Uses shadcn/ui components (`Button`, `Card`, `Input`, `Label`, `Select`, `Badge`, `Table`) built on top of Radix UI primitives. Styled with Tailwind CSS.


## 11. Module Dependency Map

```
                    src/utils.py
                   /      |      \
                  /       |       \
  src/preprocessing.py    |    src/feature_engineering.py
         \                |                /
          \               |               /
           \         src/evaluate.py     /
            \              |            /
             \             |           /
              \       src/train.py    /
               \           |        /
                \          |       /
                 \    src/predict.py
                  \        |
                   \       |
                    main.py (FastAPI)
                        |
                   frontend/ (Next.js)
```

**Import rules:**
- `utils.py` imports nothing from `src/` (dependency-free base)
- `preprocessing.py` imports only `utils`
- `feature_engineering.py` imports only `utils`
- `evaluate.py` imports `utils`
- `train.py` imports all four above modules + `visualization`
- `predict.py` imports `utils` and `feature_engineering` (needs `engineer_features` at inference time)
- `main.py` imports `predict` and `utils`


## 12. End-to-End Smoke Test

One final cell runs the complete pipeline from raw data through prediction and verifies metrics match training.


In [ ]:
from src.utils import load_raw_data
from src.preprocessing import clean_dataframe, split_data
from src.feature_engineering import engineer_features, get_all_feature_groups
from src.evaluate import compute_metrics
from src.predict import predict_charges, clear_model_cache

# Full pipeline check
raw      = load_raw_data()
cleaned  = clean_dataframe(raw)
enriched = engineer_features(cleaned)
_, X_test, _, y_test = split_data(enriched)

# Verify model prediction
clear_model_cache()
sample = predict_charges({'age':35,'sex':'male','bmi':28.5,'children':2,'smoker':'no','region':'northwest'})
print(f'Sample prediction: ${sample:,.2f}')

# Verify metrics still match expected
import joblib
from src.utils import MODELS_DIR
bundle  = joblib.load(MODELS_DIR / 'best_model.pkl')
num_f, cat_f = get_all_feature_groups(enriched.drop(columns=['charges']))

import numpy as np
y_pred = bundle['model'].predict(X_test)
metrics = compute_metrics(y_test.values, y_pred)
print(f'RMSE: ${metrics["rmse"]:,.0f}  MAE: ${metrics["mae"]:,.0f}  R²: {metrics["r2"]:.4f}')
print('\n✅ All modules working end-to-end')
